Step 1: Import Libraries & Load Data

In [0]:
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import issparse
import os

# Block 1: Import Libraries and Load Your Data

import joblib
import numpy as np
from scipy.sparse import issparse

# Load transformed feature pipeline (optional, if used later)
pipeline = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/stedi_feature_pipeline.pkl")

# Load transformed datasets
X_train_transformed = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/X_train_transformed.pkl")
X_test_transformed = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/X_test_transformed.pkl")
y_train = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/y_train.pkl")
y_test = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/y_test.pkl")

# Flatten y labels just in case
y_train = np.ravel(y_train)
y_test = np.ravel(y_test)

# Slice X to match y dimensions
n_train_samples = y_train.shape[0]
n_test_samples = y_test.shape[0]

X_train = X_train_transformed[:n_train_samples, :]
X_test = X_test_transformed[:n_test_samples, :]

# Confirm dimensions match
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

# Safety assertions (fail fast if mismatched)
assert X_train.shape[0] == y_train.shape[0], "Training data mismatch!"
assert X_test.shape[0] == y_test.shape[0], "Test data mismatch!"


Step 2: Logistic Regression Tuning

In [0]:
# Block 2: Logistic Regression Tuning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

log_reg_params = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["lbfgs", "liblinear"]
}

log_reg_grid = GridSearchCV(
    LogisticRegression(max_iter=300),
    log_reg_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

log_reg_grid.fit(X_train, y_train)

log_reg_best_params = log_reg_grid.best_params_
log_reg_best_score = log_reg_grid.best_score_

log_reg_best_params, log_reg_best_score

Step 3: Random Forest Tuning

In [0]:
# Block 3: Random Forest Tuning
from sklearn.ensemble import RandomForestClassifier

rf_params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(),
    rf_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1  # Use all cores
)

rf_grid.fit(X_train, y_train)

rf_best_params = rf_grid.best_params_
rf_best_score = rf_grid.best_score_

rf_best_params, rf_best_score

Step 4: Compare Tuned Models

In [0]:
results = {
    "Logistic Regression (tuned)": log_reg_best_score,
    "Random Forest (tuned)": rf_best_score
}
results

Step 5: Select & Save Best Model

In [0]:
import os
import joblib

# Determine best model
if rf_best_score > log_reg_best_score:
    best_model = rf_grid.best_estimator_
    best_model_name = "Random Forest"
else:
    best_model = log_reg_grid.best_estimator_
    best_model_name = "Logistic Regression"

print(f"Best model: {best_model_name}")
print(best_model)

# Define save path in GitHub Repo (Databricks Repos path)
repo_model_path = "/Workspace/Repos/win185@ensign.edu/Databricks/models/stedi_best_model.pkl"

# Save model to the repo path
joblib.dump(best_model, repo_model_path)

# Confirm it saved
os.path.exists(repo_model_path)

Step 6: Evaluation & Ethics Reflection

### Model Evaluation Report

Both Logistic Regression and Random Forest achieved the exact same cross-validation accuracy of **95.11%**. This indicates that the models performed equally well on the training data with respect to accuracy. However, when models tie in performance, other factors such as interpretability, computational efficiency, and downstream usage become important.

Logistic Regression was selected as the final model because it is simpler, faster to train, and easier to interpret. These advantages make it more suitable for future work such as SHAP explanations, dashboarding, and communicating results to stakeholders. Additionally, its lower risk of overfitting and transparency make it a reliable baseline for further development.

If more time were available, I would explore other model types such as Gradient Boosted Trees (e.g., XGBoost) or ensemble methods, and potentially test additional hyperparameter combinations. I would also evaluate the models using precision, recall, and confusion matrices to ensure balanced performance across all classes.


### Ethics and Fairness Reflection

While hyperparameter tuning helps optimize model performance, it can also unintentionally introduce **bias**. For example, tuning solely for accuracy may lead the model to favor the majority class, ignoring fairness for underrepresented groups. Similarly, complex models like Random Forest can obscure the reasons behind predictions, making it harder to detect bias.

This highlights the importance of **transparency** in machine learning. By clearly documenting models, parameters, and decisions, we allow others to evaluate our work honestly. This aligns with the gospel principle in **Alma 37:6**, which teaches that "by small and simple things are great things brought to pass." Just as small, intentional steps in our spiritual lives bring lasting growth, careful and honest evaluation of models — even the “small” decisions — contributes to ethical, trustworthy AI systems.

As data scientists, we must strive not only for performance, but for fairness, explainability, and accountability.